In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt

# Load CIFAR-10 dataset
cifar_data = torchvision.datasets.CIFAR10(
    root='./data',
    train=True,
    download=True,
    transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0,], std=[1])
])# Converts to tensor and normalizes to [0, 1]
)

print(f"CIFAR-10 loaded: {len(cifar_data)}images")
print(f"Image shape: {cifar_data[0][0].shape}") # Should be (3, 32, 32)

In [ ]:
from PIL import Image


class ColorizationDataset(Dataset):
    """
    A dataset for image colorization.
    Returns (grayscale_image, color_image) pairs.

    Args:
        cifar_dataset: The CIFAR-10 dataset (already transformed to tensors)
    """
    def __init__(self, cifar_data, transform_gray, transform_color):
        self.cifar_data = cifar_data
        self.transform_gray = transform_gray
        self.transform_color = transform_color
        pass

    def __len__(self):
        return len(self.cifar_data)
        pass

    def rgb_to_grayscale(self, img):
        """
        Convert an RGB image to grayscale.

        Args:
            img: Tensor of shape (3, H, W) with values in [0, 1]

        Returns:
            Tensor of shape (1, H, W) with values in [0, 1]
        """
        # TO DO: Implement RGB to grayscale conversion
        # Hint: Gray = 0.299 * R + 0.587 * G + 0.114 * B

        transform_gray=cifar_data.reshape(0.299*1,0.587*230,0.114*255)
        pass

    def __getitem__(self, idx):
        img = Image.open(self.image_paths[idx]).convert("RGB")

        gray = img.convert("L")
        gray = self.transform_gray(gray)
        color = self.transform_color(img)
        # TO DO: Get the color image and convert to grayscale
        # Return (grayscale_image, color_image)
        pass

In [ ]:
# Test your implementation
colorization_dataset = ColorizationDataset(cifar_data)

# Get a sample
gray_img, color_img = colorization_dataset[0]

print(f"Grayscale image shape: {gray_img.shape}")  # Should be (1, 32, 32)
print(f"Color image shape: {color_img.shape}")      # Should be (3, 32, 32)

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(6, 3))
axes[0].imshow(gray_img.squeeze(), cmap='gray')
axes[0].set_title('Grayscale (Input)')
axes[0].axis('off')
axes[1].imshow(color_img.permute(1, 2, 0))
axes[1].set_title('Color (Target)')
axes[1].axis('off')
plt.show()

In [ ]:
# Test with DataLoader
dataloader = DataLoader(colorization_dataset, batch_size=8, shuffle=True)

gray_batch, color_batch = next(iter(dataloader))
print(f"Batch grayscale shape: {gray_batch.shape}")  # Should be (8, 1, 32, 32)
print(f"Batch color shape: {color_batch.shape}")      # Should be (8, 3, 32, 32)

In [ ]:
# below is my intiative to do the colorization from gray to color using using autoencoder

In [ ]:
import torch
import torchvision.transforms as transforms
from torchvision.datasets import CIFAR10
from torch.utils.data import DataLoader
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import glob
import os

# Define transformations (convert to tensor)                         ## Will study in-depth in a next lab
transform = transforms.Compose([
    transforms.ToTensor(),
])

# Load CIFAR-10 dataset
train_dataset = CIFAR10(root="./data", train=True, transform=transform, download=True)
test_dataset = CIFAR10(root="./data", train=False, transform=transform, download=True)

# Create DataLoaders
# (The Dataset Class loads only one sample at a time. We pass it to dataloader to read batch_size of images at a time)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=2)

# Check dataset size
print(f"Training samples: {len(train_dataset)}, Testing samples: {len(test_dataset)}")

In [ ]:
transform_gray = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor()
])

transform_color = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor()
])

class DecolorizationAE(nn.Module):
    def __init__(self):
        super().__init__()

        self.encoder = nn.Sequential(
            nn.Conv2d(3, 32, 3, stride=2, padding=1), # Changed input to 3 channels
            nn.ReLU(),
            nn.Conv2d(32, 64, 3, stride=2, padding=1),
            nn.ReLU()
        )

        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(64, 32, 3, stride=2, padding=1, output_padding=1),
            nn.ReLU(),
            nn.ConvTranspose2d(32, 1, 3, stride=2, padding=1, output_padding=1), # Changed output to 1 channel
            nn.Sigmoid()
        )

    def forward(self, x):
        z = self.encoder(x)
        out = self.decoder(z)
        return out

In [ ]:
# Initialize the model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = ColorizationAE().to(device)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

In [ ]:
for epoch in range(10):
    for gray, color in train_loader:
      gray, color = gray.to(device), color.to(device)

      output = model(color)
      loss = criterion(output, gray)

      optimizer.zero_grad()
      loss.backward()
      optimizer.step()